In [55]:
# Instalar librerías

!pip install -q mistralai
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-huggingface
!pip install -q chromadb
!pip install -q sentence-transformers

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-sdk 1.42.1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.39.1 which is incompatible.
opentelemetry-sdk 1.42.1 requires opentelemetry-semantic-conventions==0.63b1, but you have opentelemetry-semantic-conventions 0.60b1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.39.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.42.1 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.42.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-

In [56]:
# Importaciones principales

import pandas as pd
import numpy as np

from google.colab import files
from google.colab import userdata

from mistralai.client import Mistral

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score

In [57]:
# Configurar API key de Mistral

api_key = userdata.get("MistralProyecto")

client = Mistral(
    api_key=api_key
)

print("API configurada correctamente")

API configurada correctamente


In [90]:
# Cargar dataset local

uploaded = files.upload()

archivo = list(uploaded.keys())[0]

# Dataset para ML
df = pd.read_csv(archivo)

# Dataset independiente para RAG
df_original = pd.read_csv(archivo)

print(df.shape)

df.head()

Saving WA_Fn-UseC_-Telco-Customer-Churn.csv to WA_Fn-UseC_-Telco-Customer-Churn (3).csv
(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [92]:
# Eliminar customerID porque no aporta al entrenamiento

if "customerID" in df.columns:

    df = df.drop(
        columns=["customerID"]
    )

print("customerID eliminado")

customerID eliminado


In [93]:
# Convertir TotalCharges a numérico

df["TotalCharges"] = pd.to_numeric(

    df["TotalCharges"],

    errors="coerce"
)

print("Conversión realizada")

Conversión realizada


In [94]:
# Imputar valores faltantes

df["TotalCharges"] = df["TotalCharges"].fillna(

    df["TotalCharges"].median()
)

print("Valores faltantes corregidos")

Valores faltantes corregidos


In [95]:
# Codificar variables categóricas

encoder = LabelEncoder()

for columna in df.select_dtypes(include="object"):

    df[columna] = encoder.fit_transform(
        df[columna]
    )

print("Variables categóricas codificadas")

Variables categóricas codificadas


In [96]:
# Escalar variables numéricas

scaler = StandardScaler()

columnas_numericas = [

    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

df[columnas_numericas] = scaler.fit_transform(

    df[columnas_numericas]
)

print("Escalado finalizado")

Escalado finalizado


In [97]:
# Verificar dataset limpio

df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,1,0,-1.277445,0,1,0,0,2,0,0,0,0,0,1,2,-1.160323,-0.994242,0
1,1,0,0,0,0.066327,1,0,0,2,0,2,0,0,0,1,0,3,-0.259629,-0.173244,0
2,1,0,0,0,-1.236724,1,0,0,2,2,0,0,0,0,0,1,3,-0.362660,-0.959674,1
3,1,0,0,0,0.514251,0,1,0,2,0,2,2,0,0,1,0,0,-0.746535,-0.194766,0
4,0,0,0,0,-1.236724,1,0,1,0,0,0,0,0,0,0,1,2,0.197365,-0.940470,1


In [98]:
# Separar variables predictoras y objetivo

X = df.drop(
    columns=["Churn"]
)

y = df["Churn"]

print("Variables separadas")
print(X.shape)

Variables separadas
(7043, 19)


In [99]:
# Dividir entrenamiento y prueba

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y
)

print("División completada")

División completada


In [100]:
# Entrenar modelo

modelo = LogisticRegression(

    max_iter=1000
)

modelo.fit(

    X_train,
    y_train
)

print("Modelo entrenado")

Modelo entrenado


In [101]:
# Generar predicciones

predicciones = modelo.predict(

    X_test
)

print("Predicciones generadas")

Predicciones generadas


In [102]:
# Accuracy

accuracy = accuracy_score(

    y_test,
    predicciones
)

print(

    f"Accuracy: {accuracy:.4f}"
)

Accuracy: 0.8006


In [103]:
# Reporte completo

print(

    classification_report(

        y_test,
        predicciones
    )
)

              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.65      0.55      0.59       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.73      1409
weighted avg       0.79      0.80      0.80      1409



In [104]:
# Agente normalizador

class AgenteNormalizador:

    def invoke(self, dataframe):

        df = dataframe.copy()

        if "customerID" in df.columns:

            df = df.drop(
                columns=["customerID"]
            )

        df["TotalCharges"] = pd.to_numeric(

            df["TotalCharges"],
            errors="coerce"
        )

        df["TotalCharges"] = df["TotalCharges"].fillna(

            df["TotalCharges"].median()
        )

        encoder = LabelEncoder()

        for columna in df.select_dtypes(include="object"):

            df[columna] = encoder.fit_transform(

                df[columna]
            )

        scaler = StandardScaler()

        columnas_numericas = [

            "tenure",
            "MonthlyCharges",
            "TotalCharges"
        ]

        df[columnas_numericas] = scaler.fit_transform(

            df[columnas_numericas]
        )

        return df

In [105]:
# Ejecutar agente

agente_normalizador = AgenteNormalizador()

df_limpio = agente_normalizador.invoke(
    df
)

print(df_limpio.head())

   gender  SeniorCitizen  Partner  Dependents    tenure  PhoneService  \
0       0              0        1           0 -1.277445             0   
1       1              0        0           0  0.066327             1   
2       1              0        0           0 -1.236724             1   
3       1              0        0           0  0.514251             0   
4       0              0        0           0 -1.236724             1   

   MultipleLines  InternetService  OnlineSecurity  OnlineBackup  \
0              1                0               0             2   
1              0                0               2             0   
2              0                0               2             2   
3              1                0               2             0   
4              0                1               0             0   

   DeviceProtection  TechSupport  StreamingTV  StreamingMovies  Contract  \
0                 0            0            0                0         0   
1     

In [106]:
# Agente entrenador

class AgenteEntrenador:

    def invoke(self, dataframe):

        X = dataframe.drop(
            columns=["Churn"]
        )

        y = dataframe["Churn"]

        X_train, X_test, y_train, y_test = train_test_split(

            X,
            y,

            test_size=0.2,

            random_state=42,

            stratify=y
        )

        modelo = LogisticRegression(

            max_iter=1000
        )

        modelo.fit(

            X_train,
            y_train
        )

        predicciones = modelo.predict(

            X_test
        )

        accuracy = accuracy_score(

            y_test,
            predicciones
        )

        metricas = classification_report(

            y_test,
            predicciones,

            output_dict=True
        )

        return {

            "modelo":modelo,

            "accuracy":accuracy,

            "metricas":metricas
        }

In [107]:
# Ejecutar agente entrenador

agente_entrenador = AgenteEntrenador()

resultado = agente_entrenador.invoke(
    df_limpio
)

In [108]:
# Visualizar resultados

print(

    f"Accuracy: {resultado['accuracy']:.4f}"
)

Accuracy: 0.8006


In [109]:
# Mostrar métricas principales

print(

    resultado["metricas"]["weighted avg"]
)

{'precision': 0.7925355765149744, 'recall': 0.8005677785663591, 'f1-score': 0.7950145047296682, 'support': 1409.0}


In [110]:
# Agente comunicador

class AgenteComunicador:

    def __init__(self, client):

        self.client = client

    def invoke(self, resultados):

        accuracy = resultados["accuracy"]

        metricas = resultados["metricas"]

        prompt = f"""
        Eres un analista de Machine Learning especializado
        en abandono de clientes.

        Reglas estrictas:

        - No utilices emojis.
        - No uses tablas.
        - No inventes variables o información inexistente.
        - No menciones características que no aparezcan
          en las métricas entregadas.
        - Mantén un tono profesional.
        - Escribe máximo 4 párrafos.
        - Explica:

        1. Accuracy general
        2. Desempeño para clientes que permanecen
        3. Desempeño para clientes que abandonan
        4. Debilidad principal del modelo

        Clase 0 = Cliente permanece
        Clase 1 = Cliente abandona

        Accuracy:

        {accuracy}

        Métricas:

        {metricas}
        """

        respuesta = self.client.chat.complete(

            model="mistral-small-latest",

            messages=[

                {
                    "role":"user",
                    "content":prompt
                }

            ]
        )

        return respuesta.choices[0].message.content

In [111]:
# Ejecutar agente comunicador

agente_comunicador = AgenteComunicador(
    client
)

reporte = agente_comunicador.invoke(
    resultado
)

In [112]:
# Mostrar reporte

print(reporte)

El modelo presenta un accuracy general del 80.06%, lo que indica que acierta correctamente en aproximadamente 8 de cada 10 predicciones. Este valor sugiere un desempeño aceptable, aunque no óptimo, en la clasificación de clientes según su estado de permanencia o abandono.

En cuanto a los clientes que permanecen (Clase 0), el modelo alcanza una precisión del 84.52% y un recall del 89.18%, con un F1-score de 86.79%. Estos resultados reflejan una alta capacidad para identificar correctamente a los clientes que no abandonan, minimizando tanto los falsos positivos como los falsos negativos en esta categoría.

Para los clientes que abandonan (Clase 1), el desempeño es notablemente inferior: la precisión es del 64.67%, el recall del 54.81% y el F1-score del 59.33%. La baja tasa de recall indica que el modelo no logra detectar más de la mitad de los casos reales de abandono, lo que sugiere una tendencia a subestimar este grupo.

La principal debilidad del modelo radica en su bajo rendimiento 

In [113]:
# Importaciones para RAG

from langchain_core.documents import Document

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import Chroma

In [114]:
# Crear documentos desde dataset original

documentos = []

for _, fila in df_original.iterrows():

    abandono = "Sí" if fila["Churn"]=="Yes" else "No"

    texto = f"""

    Cliente con {fila["tenure"]} meses de permanencia.

    Género: {fila["gender"]}

    Tiene pareja: {fila["Partner"]}

    Tiene dependientes: {fila["Dependents"]}

    Tipo de contrato: {fila["Contract"]}

    Servicio de internet: {fila["InternetService"]}

    Soporte técnico: {fila["TechSupport"]}

    Facturación electrónica: {fila["PaperlessBilling"]}

    Método de pago: {fila["PaymentMethod"]}

    Cargo mensual: {fila["MonthlyCharges"]} dólares.

    Cargo total: {fila["TotalCharges"]} dólares.

    Estado abandono: {abandono}

    """

    documentos.append(

        Document(
            page_content=texto
        )
    )

print(

    f"Documentos creados: {len(documentos)}"
)

Documentos creados: 7043


In [115]:
# Modelo de embeddings

embeddings = HuggingFaceEmbeddings(

    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings configurados")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings configurados


In [121]:
# Crear base vectorial limpia

vector_db = Chroma.from_documents(

    documents=documentos,

    embedding=embeddings,

    collection_name="telco_rag_nuevo"
)

print("Base vectorial recreada")

Base vectorial recreada


In [122]:
# Configurar retriever

retriever = vector_db.as_retriever(

    search_kwargs={

        "k":15
    }
)

print(

    "Retriever configurado"
)

Retriever configurado


In [123]:
# Prueba recuperación semántica

consulta = "clientes que abandonan"

resultados = retriever.invoke(
    consulta
)

for i, doc in enumerate(resultados):

    print()

    print("="*50)

    print(f"Resultado {i+1}")

    print(doc.page_content)


Resultado 1

    
    Cliente con 1 meses de permanencia.

    Género: Male

    Tiene pareja: No

    Tiene dependientes: No

    Tipo de contrato: Month-to-month

    Servicio de internet: No

    Soporte técnico: No internet service

    Facturación electrónica: No

    Método de pago: Mailed check

    Cargo mensual: 20.2 dólares.

    Cargo total: 20.2 dólares.

    Estado abandono: Sí
    
    

Resultado 2

    
    Cliente con 1 meses de permanencia.

    Género: Male

    Tiene pareja: No

    Tiene dependientes: No

    Tipo de contrato: Month-to-month

    Servicio de internet: No

    Soporte técnico: No internet service

    Facturación electrónica: No

    Método de pago: Mailed check

    Cargo mensual: 20.05 dólares.

    Cargo total: 20.05 dólares.

    Estado abandono: Sí
    
    

Resultado 3

    
    Cliente con 1 meses de permanencia.

    Género: Male

    Tiene pareja: No

    Tiene dependientes: No

    Tipo de contrato: Month-to-month

    Servicio de intern

In [126]:
# Agente comunicador con RAG

class AgenteComunicadorRAG:

    def __init__(self, client, retriever):

        self.client = client
        self.retriever = retriever


    def invoke(self, pregunta):

        documentos = self.retriever.invoke(
            pregunta
        )

        contexto = "\n\n".join(

            [

                doc.page_content

                for doc in documentos

            ]
        )

        prompt = f"""
        Eres un asistente especializado en análisis
        de abandono de clientes.

        Reglas:

        - Responde únicamente usando el contexto.
        - No inventes información.
        - No uses emojis.
        - Usa lenguaje profesional.
        - Si no encuentras información suficiente,
          indícalo.

        Contexto:

        {contexto}

        Pregunta:

        {pregunta}
        """

        respuesta = self.client.chat.complete(

            model="mistral-small-latest",

            messages=[

                {

                    "role":"user",

                    "content":prompt
                }

            ]
        )

        return respuesta.choices[0].message.content

In [127]:
# Inicializar agente

agente_rag = AgenteComunicadorRAG(

    client,
    retriever
)

print("Agente RAG listo")

Agente RAG listo


In [132]:
# Primera consulta

respuesta = agente_rag.invoke(

    "¿Qué característica clave tienen los clientes que abandonan?"
)

print(respuesta)

Basado en el contexto proporcionado, la característica clave de los clientes que abandonan es el tipo de contrato. Todos los clientes con estado de abandono tienen un contrato de tipo *Month-to-month*, mientras que el único cliente sin abandono tiene un contrato *Two year*.

No se observan diferencias significativas en las demás variables (género, estado civil, dependientes, servicio de internet, soporte técnico, facturación electrónica o método de pago) que permitan identificar otro patrón relevante.


In [134]:
# Segunda consulta

respuesta = agente_rag.invoke(

    "¿Qué tipo de contrato tienen los clientes que permanecen?"
)

print(respuesta)

Los clientes que permanecen tienen un contrato de tipo *Month-to-month*.


In [131]:
# Tercera consulta

respuesta = agente_rag.invoke(

    "¿Qué métodos de pago aparecen con frecuencia?"
)

print(respuesta)

Los métodos de pago que aparecen con frecuencia en el contexto proporcionado son:

- **Mailed check** (cheque enviado por correo).


In [135]:
# Sistema multiagente completo

class SistemaMultiAgente:

    def __init__(

        self,

        normalizador,

        entrenador,

        comunicador

    ):

        self.normalizador = normalizador

        self.entrenador = entrenador

        self.comunicador = comunicador


    def invoke(self, dataframe, pregunta):

        datos_limpios = self.normalizador.invoke(

            dataframe
        )

        resultados = self.entrenador.invoke(

            datos_limpios
        )

        respuesta = self.comunicador.invoke(

            pregunta
        )

        return {

            "accuracy":resultados["accuracy"],

            "respuesta":respuesta
        }

In [136]:
# Crear sistema completo

sistema = SistemaMultiAgente(

    agente_normalizador,

    agente_entrenador,

    agente_rag
)

print(

    "Sistema multiagente listo"
)

Sistema multiagente listo


In [137]:
# Ejecutar sistema completo

resultado_final = sistema.invoke(

    df,

    "¿Qué características tienen los clientes que abandonan?"
)

In [138]:
# Mostrar resultados

print(

    f"Accuracy: {resultado_final['accuracy']:.4f}"
)

print()

print(

    resultado_final["respuesta"]
)

Accuracy: 0.8006

Basado en el contexto proporcionado, los clientes que abandonan comparten las siguientes características:

1. **Tipo de contrato**: Todos los clientes que abandonan tienen un contrato *Month-to-Month* (mes a mes), excepto un cliente con contrato *Two year* que no abandonó.

2. **Servicio de internet**: Ninguno de los clientes que abandonan tiene servicio de internet.

3. **Soporte técnico**: Todos reportan *No internet service* (sin servicio de internet).

4. **Facturación electrónica**: La mayoría no tiene facturación electrónica, excepto un cliente que sí la tiene y abandonó.

5. **Método de pago**: Todos los clientes que abandonan usan *Mailed check* (pago por cheque enviado por correo).

6. **Cargo mensual**: Los cargos mensuales varían entre **20.15 y 20.9 dólares**, sin un patrón claro de abandono asociado a un monto específico.

7. **Género**: Hay una distribución equilibrada entre hombres y mujeres que abandonan.

8. **Situación familiar**: Todos los clientes 

In [139]:
# Flujo principal del sistema

pregunta_usuario = """
¿Qué características presentan los clientes
que abandonan el servicio?
"""

resultado_sistema = sistema.invoke(

    df,

    pregunta_usuario
)

print(

    f"Accuracy del modelo: {resultado_sistema['accuracy']:.4f}"
)

print()

print(

    resultado_sistema["respuesta"]
)

Accuracy del modelo: 0.8006

Basado en el contexto proporcionado, los clientes que abandonan el servicio (estado de abandono: **Sí**) presentan las siguientes características comunes:

1. **Género**: Todos son hombres.
2. **Estado civil**: No tienen pareja.
3. **Dependientes**: No tienen dependientes.
4. **Tipo de contrato**: Todos tienen contrato *Month-to-month* (mes a mes), excepto un cliente con contrato *Two year* que no abandona.
5. **Servicio de internet**: Ninguno tiene servicio de internet.
6. **Soporte técnico**: Todos reportan "No internet service".
7. **Facturación electrónica**: Ninguno utiliza facturación electrónica.
8. **Método de pago**: Todos pagan mediante cheque enviado por correo (*Mailed check*).
9. **Cargo mensual**: Los montos varían entre **20.05 y 20.5 dólares**, con una tendencia a abandonar en rangos cercanos a **20.15-20.25 dólares**.

**Observación**:
- El único cliente con contrato *Two year* no abandona, lo que sugiere que la flexibilidad del contrato po